### Estimating Dipole Moment of SO+

Using the pyscf and pyberny packages. Code generated by Gemini Pro 3, edited and commented by BAM.

In [1]:
import pyscf
from pyscf import dft
from pyscf.geomopt.berny_solver import optimize 
from pyscf.hessian import thermo

In [2]:
#Define the SO+ molecule
mol = pyscf.M(
    atom='S 0.0 0.0 0.0; O 0.0 0.0 1.424',
    basis='6-311++g(d,p)',
    spin=1,
    charge=1,
    unit='Angstrom'
)

In [3]:
#Build the Unrestricted DFT object (UKS) and set the functional
mf = dft.UKS(mol)
mf.xc = 'm06-2x'

In [4]:
#Perform the Geometry Optimization
print("--- Starting Geometry Optimization (M06-2X / 6-311++G(d,p)) ---")
mol_eq = optimize(mf)

--- Starting Geometry Optimization (M06-2X / 6-311++G(d,p)) ---

Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   S   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   O   0.000000   0.000000   1.424000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -472.946645843906  <S^2> = 0.7532055  2S+1 = 2.0032029
--------------- UKS_Scanner gradients ---------------
         x                y                z
0 S    -0.0000000000    -0.0000000000    -0.0082205948
1 O     0.0000000000     0.0000000000     0.0084013924
----------------------------------------------
cycle 1: E = -472.946645844  dE = -472.947  norm(grad) = 0.0117542

Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   S   0.000000   0.000000   0.004055    0.000000  0.000000  0.004055
   O   0.000000   0.0

In [5]:
print("\nOptimized Geometry Structure:")
print(mol_eq.tostring())


Optimized Geometry Structure:
S           0.00000000        0.00000000        0.00264242
O           0.00000000        0.00000000        1.42135758


In [6]:
#Run a final single-point calculation on the optimized structure.
# 'optimize()' returns a new Mole object (mol_eq) with updated coordinates.
print("\n--- Running Final Calculation on Optimized Geometry ---")
mf_eq = dft.UKS(mol_eq)
mf_eq.xc = 'm06-2x'
mf_eq.kernel()


--- Running Final Calculation on Optimized Geometry ---
converged SCF energy = -472.946689493088  <S^2> = 0.75317744  2S+1 = 2.0031749


-472.94668949308834

In [7]:
#Extract and print the final dipole moment
print("\n--- Final Dipole Moment Results ---")
dipole_vector = mf_eq.dip_moment()


--- Final Dipole Moment Results ---

WARN: System has nonzero charge 1; the dipole moment is origin-dependent.
Location of origin: [0. 0. 0.]

Dipole moment(X, Y, Z, Debye):  0.00000, -0.00000, -0.20796


In [8]:
#Compute Rotational Constants for comparison to lab work (B = 23249.1 from Amano:1991:519)
print("\n=== ROTATIONAL CONSTANT ===")
# PySCF stores coordinates in Bohr internally; thermo.rotation_const expects Bohr and AMU
masses = mol_eq.atom_mass_list()
coords = mol_eq.atom_coords()

# Calculate the constants in MHz
rot_constants = thermo.rotation_const(masses, coords, unit='GHz')*1000.
print(f'A: {rot_constants[0]:.2f} MHz\nB: {rot_constants[1]:.2f} MHz\nC: {rot_constants[2]:.2f} MHz')


=== ROTATIONAL CONSTANT ===
A: inf MHz
B: 23539.54 MHz
C: 23539.54 MHz


Given that these are B_e values, this is close enough to the Amano work for the dipole to be a reasonable estimate.